In [3]:
import asyncio
import subprocess
import requests
from tapo import ApiClient

# ===== 設定 =====
USERNAME = "straydog12341234@gmail.com"
PASSWORD = "d958b40323"

TAPO_IP = "192.168.0.75"

# HR01 USB-LAN
HR01_INTERFACE = "enxc436c0ebd4ef"

# HR01ゲートウェイ
HR01_GATEWAY = "192.168.128.1"

# Squid
SQUID_PROXY = "http://127.0.0.1:3128"
# =================


def get_global_ip(interface):
    """指定インターフェースから直接グローバルIP取得"""

    result = subprocess.run(
        [
            "curl",
            "--interface",
            interface,
            "--silent",
            "--max-time",
            "10",
            "https://api.ipify.org"
        ],
        capture_output=True,
        text=True,
        check=True
    )

    return result.stdout.strip()



def reset_hr01_route():
    """HR01ルーティング再設定"""

    print("\n--- HR01ルート再設定 ---")

    subprocess.run(
        [
            "sudo",
            "ip",
            "route",
            "replace",
            "default",
            "via",
            HR01_GATEWAY,
            "dev",
            HR01_INTERFACE,
            "table",
            "hr01"
        ],
        check=True
    )


    result = subprocess.run(
        [
            "ip",
            "route",
            "show",
            "table",
            "hr01"
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)



def get_squid_ip():
    """Squid経由のグローバルIP取得"""

    proxies = {
        "http": SQUID_PROXY,
        "https": SQUID_PROXY,
    }


    r = requests.get(
        "https://api.ipify.org",
        proxies=proxies,
        timeout=10
    )

    return r.text.strip()



async def main():

    client = ApiClient(USERNAME, PASSWORD)

    device = await client.p110(TAPO_IP)


    # =========================
    # HR01再起動
    # =========================

    print("HR01 電源OFF")
    await device.off()


    print("5秒待機...")
    await asyncio.sleep(5)


    print("HR01 電源ON")
    await device.on()


    print("HR01起動待ち 60秒...")
    await asyncio.sleep(60)



    # =========================
    # ルーティング復旧
    # =========================

    try:
        reset_hr01_route()

    except Exception as e:
        print("HR01ルート設定失敗")
        print(e)



    # =========================
    # HR01直接確認
    # =========================

    print("\n--- HR01直接通信確認 ---")


    try:

        hr01_ip = get_global_ip(HR01_INTERFACE)

        print(f"HR01直接IP : {hr01_ip}")

    except Exception as e:

        print("HR01直接取得失敗")
        print(e)

        hr01_ip = None



    # =========================
    # Squid確認
    # =========================

    print("\n--- Squid経由確認 ---")


    try:

        squid_ip = get_squid_ip()

        print(f"Squid経由IP : {squid_ip}")


    except Exception as e:

        print("Squid取得失敗")
        print(e)

        squid_ip = None



    # =========================
    # 比較
    # =========================

    print("\n--- 結果 ---")


    if hr01_ip and squid_ip:

        if hr01_ip == squid_ip:

            print("成功: SquidはHR01経由です")

        else:

            print("注意: IPが違います")
            print("HR01 :", hr01_ip)
            print("Squid:", squid_ip)



    print("\n処理完了")



# Jupyter Notebook
await main()


# .py実行の場合
# if __name__ == "__main__":
#     asyncio.run(main())

HR01 電源OFF
5秒待機...
HR01 電源ON
HR01起動待ち 60秒...

--- HR01ルート再設定 ---
HR01ルート設定失敗
Command '['sudo', 'ip', 'route', 'replace', 'default', 'via', '192.168.128.1', 'dev', 'enxc436c0ebd4ef', 'table', 'hr01']' returned non-zero exit status 2.

--- HR01直接通信確認 ---


Error: Nexthop has invalid gateway.


HR01直接取得失敗
Command '['curl', '--interface', 'enxc436c0ebd4ef', '--silent', '--max-time', '10', 'https://api.ipify.org']' returned non-zero exit status 28.

--- Squid経由確認 ---
Squid経由IP : 219.105.53.125

--- 結果 ---

処理完了
